# Stage 0 Loading Data
- Outcome: Load whole dataset as `df`

In [1]:
import pandas as pd 
path = 'Data' # Path of the dataset folder
df = pd.read_csv(path + '/train/train_data_ads.csv')
print("data is loaded...")

data is loaded...


# Stage 1 Pick a Advertisement Task

- Outcome: 
    - Pick one task and load it as `df_task`
    - Drop the Columns with unique value > 1000 or non-numerical value.

In [47]:
task_id = 22280 # Pick a task
df_task = df[df['task_id'] == task_id]

In [48]:
import numpy as np
# Importing the NumPy library, which provides support for large, multi-dimensional arrays and matrices,
# along with a large collection of high-level mathematical functions to operate on these arrays.

columns_to_drop = [column for column in df_task.columns if df_task[column].nunique() > 1000 or df_task[column].nunique() == 1]
# Constructing a list of columns from the dataframe 'df_task' to be dropped.
# A column is added to this list if it has more than 1000 unique values, which typically
# suggests that the column contains highly granular data, possibly not useful for analysis
# or could lead to issues like overfitting if used in machine learning models.

df_task = df_task.drop(columns=columns_to_drop)
# Removing the columns identified in the 'columns_to_drop' list from 'df_task'.
# This operation simplifies the dataframe by excluding columns with excessive uniqueness.

df_task = df_task.select_dtypes(include=[np.number])
# Filtering the dataframe to include only columns that have numerical data types.
# This step is crucial for analyses that require numerical inputs, such as mathematical
# operations or statistical modeling.

print(df_task.dtypes)
# Printing the data types of the columns remaining in the dataframe 'df_task'.
# This is useful for verifying that the dataframe now contains only numerical columns,
# as intended after the previous filtering step.

label              int64
age                int64
gender             int64
residence          int64
city               int64
city_rank          int64
series_dev         int64
series_group       int64
emui_dev           int64
device_name        int64
device_size        int64
creat_type_cd      int64
slot_id            int64
u_refreshTimes     int64
u_feedLifeCycle    int64
dtype: object


# Stage 2 Break Dataset into Training, Holdout, Validate
- Outcome: `df_task_train`, `df_task_holdout`, `df_task_val`

## Step 2.1 Check Positive Sample Size, Negative Sample Size, Label Rate

In [49]:
df_label_0 = df_task[df_task['label'] == 0]
df_label_1 = df_task[df_task['label'] == 1]

# 對 label 為 0 的數據集進行分割
total_samples_label_0 = len(df_label_0)
train_size_label_0 = int(0.4 * total_samples_label_0)
holdout_size_label_0 = int(0.4 * total_samples_label_0)
validate_size_label_0 = total_samples_label_0 - train_size_label_0 - holdout_size_label_0

df_label_0_train = df_label_0.sample(n=train_size_label_0, random_state=42)
df_label_0_hold = df_label_0.drop(df_label_0_train.index).sample(n=holdout_size_label_0, random_state=42)
df_label_0_val = df_label_0.drop(df_label_0_train.index).drop(df_label_0_hold.index)

# 對 label 為 1 的數據集進行分割
total_samples_label_1 = len(df_label_1)
train_size_label_1 = int(0.4 * total_samples_label_1)
holdout_size_label_1 = int(0.4 * total_samples_label_1)
validate_size_label_1 = total_samples_label_1 - train_size_label_1 - holdout_size_label_1

df_label_1_train = df_label_1.sample(n=train_size_label_1, random_state=42)
df_label_1_hold = df_label_1.drop(df_label_1_train.index).sample(n=holdout_size_label_1, random_state=42)
df_label_1_val = df_label_1.drop(df_label_1_train.index).drop(df_label_1_hold.index)

In [50]:
# 將 label 為 0 和 label 為 1 的 train 子數據集合併成 df_train
df_train = pd.concat([df_label_0_train, df_label_1_train])

# 將 label 為 0 和 label 為 1 的 holdout 子數據集合併成 df_holdout
df_holdout = pd.concat([df_label_0_hold, df_label_1_hold])

# 將 label 為 0 和 label 為 1 的 validate 子數據集合併成 df_val
df_val = pd.concat([df_label_0_val, df_label_1_val])

In [51]:
from Dataset_Utility.utility_functions import calculate_label_rate
calculate_label_rate(df_task)
calculate_label_rate(df_train)
calculate_label_rate(df_holdout)
calculate_label_rate(df_val)

Total Sample size is 21279, Positive Sample size is 885, Negative Sample size is 20394, label rate is 0.04
Total Sample size is 8511, Positive Sample size is 354, Negative Sample size is 8157, label rate is 0.04
Total Sample size is 8511, Positive Sample size is 354, Negative Sample size is 8157, label rate is 0.04
Total Sample size is 4257, Positive Sample size is 177, Negative Sample size is 4080, label rate is 0.04


# Stage 3 Save the three sample dataset

In [52]:
# 將df_train的行隨機重新排序
df_train_shuffled = df_train.sample(frac=1).reset_index(drop=True)

# 將df_holdout的行隨機重新排序
df_holdout_shuffled = df_holdout.sample(frac=1).reset_index(drop=True)

# 將df_val的行隨機重新排序
df_val_shuffled = df_val.sample(frac=1).reset_index(drop=True)

calculate_label_rate(df_train_shuffled)
calculate_label_rate(df_holdout_shuffled)
calculate_label_rate(df_val_shuffled)

Total Sample size is 8511, Positive Sample size is 354, Negative Sample size is 8157, label rate is 0.04
Total Sample size is 8511, Positive Sample size is 354, Negative Sample size is 8157, label rate is 0.04
Total Sample size is 4257, Positive Sample size is 177, Negative Sample size is 4080, label rate is 0.04


In [53]:
# Creat a folder to save df_train, df_holdout and df_val
import os
task_path = f'{path}/task_{task_id}'
if not os.path.exists(task_path):
    os.makedirs(task_path)

df_train_shuffled.to_csv(f'{path}/task_{task_id}/df_{task_id}_train.csv', index=False)
df_holdout_shuffled.to_csv(f'{path}/task_{task_id}/df_{task_id}_holdout.csv', index=False)
df_val_shuffled.to_csv(f'{path}/task_{task_id}/df_{task_id}_val.csv', index=False)